# 02 · Results: snow at Matsch

Results of the port on the two Matsch cases, with the C++ outputs overlaid.
`P2` (1549 m) and `B2` (1480 m) share period and forcing format: October
2009, 15-minute step.

October snow at Matsch is thin: at most 6.6 mm of depth in P2, and in B2 it
melts as it falls. For a real snowpack the notebook ends with `ARF_1D`
(69° N, January 2000, 17 cm of snow).

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd
from _helpers import (
    compare_with_reference,
    point_df,
    prepare_case,
    profile_df,
    reference_dir,
    run_case,
)

In [ ]:
CASES = ["Matsch_P2_Ref_007", "Matsch_B2_Ref_007", "ARF_1D"]
out = {}
for name in CASES:
    case = prepare_case(name)
    t0 = time.perf_counter()
    run_case(case)
    out[name] = case / "output-tabs"
    print(f"{name:20s} {time.perf_counter() - t0:5.1f} s")

Every table of the three cases against the C++ reference (absolute or relative error ≤ 1e-5):

In [ ]:
check = {name: compare_with_reference(name, out[name]) for name in CASES}
pd.DataFrame({name: {"tables": len(c), "columns": c["columns"].sum(),
                     "failing columns": c["failing"].sum(),
                     "all ok": bool((c["status"] == "ok").all())}
              for name, c in check.items()}).T

## P2 and B2: forcing and snow

In [ ]:
def pair(name, table="point0001.txt"):
    return point_df(out[name] / table), point_df(reference_dir(name) / table)

pt = {name: pair(name) for name in CASES[:2]}
cols = ["Psnow_over_canopy[mm]", "Prain_over_canopy[mm]", "Tair[C]",
        "snow_depth[mm]", "snow_water_equivalent[mm]", "Tsurface[C]"]
fig, axes = plt.subplots(len(cols), 1, figsize=(11, 13), sharex=True)
for ax, col in zip(axes, cols):
    for name, (py, ref) in pt.items():
        label = name.split("_")[1]
        line, = ax.plot(py.index, py[col], lw=1, label=f"{label} GEOtopPY")
        ax.plot(ref.index, ref[col], lw=3, alpha=0.3, color=line.get_color(),
                label=f"{label} C++")
    ax.set_ylabel(col, fontsize=8)
axes[0].legend(ncol=4, fontsize=8)
fig.tight_layout()

Zoom on the largest snow event in P2, on 1 November:

In [ ]:
py, ref = pt["Matsch_P2_Ref_007"]
peak = py["snow_depth[mm]"].idxmax()
window = slice(peak - pd.Timedelta("18h"), peak + pd.Timedelta("18h"))
print("deepest snow:", py.loc[peak, "snow_depth[mm]"], "mm at", peak)

cols = ["snow_depth[mm]", "snow_water_equivalent[mm]", "snow_density[kg/m3]",
        "snow_temperature[C]", "snow_melted[mm]"]
fig, axes = plt.subplots(len(cols), 1, figsize=(11, 10), sharex=True)
for ax, col in zip(axes, cols):
    ax.plot(ref.loc[window].index, ref.loc[window, col], lw=3, alpha=0.4, label="C++")
    ax.plot(py.loc[window].index, py.loc[window, col], lw=1, label="GEOtopPY")
    ax.set_ylabel(col, fontsize=8)
axes[0].legend()
fig.tight_layout()

## P2 profiles

Snow profile files have one column per layer (`L1` at the base), soil files
one column per depth [mm]. GEOtop's -9999 (layer absent) is NaN. At Matsch
the snow never has more than a few layers, so they are plotted as lines over
the event.

In [ ]:
p2 = out["Matsch_P2_Ref_007"]
p2_ref = reference_dir("Matsch_P2_Ref_007")
files = [("snowDepth0001.txt", "layer thickness [mm]"), ("snowT0001.txt", "layer temperature [C]"),
         ("snowLiq0001.txt", "layer liquid water [kg/m2]"), ("snowIce0001.txt", "layer ice [kg/m2]")]
fig, axes = plt.subplots(len(files), 1, figsize=(11, 9), sharex=True)
for ax, (f, label) in zip(axes, files):
    got = profile_df(p2 / f).loc[window].dropna(axis=1, how="all")
    ref = profile_df(p2_ref / f).loc[window][got.columns]
    for layer in got.columns:
        line, = ax.plot(got.index, got[layer], "o-", ms=3, lw=1, label=f"{layer} GEOtopPY")
        ax.plot(ref.index, ref[layer], lw=4, alpha=0.3, color=line.get_color())
    ax.set_ylabel(label, fontsize=8)
axes[0].legend(fontsize=8)
axes[0].set_title("thin: GEOtopPY, thick: C++", fontsize=9)
fig.tight_layout()

In [ ]:
def depth_heatmap(ax, df, title, cmap, max_depth=500.0):
    df = df[[d for d in df.columns if d <= max_depth]]
    m = ax.pcolormesh(df.index, df.columns, df.T.values, shading="nearest", cmap=cmap)
    ax.invert_yaxis()
    ax.set_ylabel("depth [mm]")
    ax.set_title(title, fontsize=9)
    plt.colorbar(m, ax=ax)

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
depth_heatmap(axes[0], profile_df(p2 / "soilTz0001.txt"), "soil temperature [C]", "coolwarm")
depth_heatmap(axes[1], profile_df(p2 / "thetaliq0001.txt"), "liquid water content [-]", "Blues")
depth_heatmap(axes[2], profile_df(p2 / "thetaice0001.txt"), "ice content [-]", "Purples")
fig.tight_layout()

Python against C++ on the soil temperature profile: largest absolute difference per depth [C].

In [ ]:
got = profile_df(p2 / "soilTz0001.txt")
ref = profile_df(reference_dir("Matsch_P2_Ref_007") / "soilTz0001.txt")
(got - ref).abs().max().to_frame("max |Python - C++|").T

## P2 surface energy balance

From `basin.txt`, written for the whole (single-point) domain.

In [ ]:
basin_py, basin_ref = pair("Matsch_P2_Ref_007", "basin.txt")
cols = ["SW[W/m2]", "LW[W/m2]", "H[W/m2]", "LE[W/m2]"]
daily = basin_py[cols].resample("1D").mean()
ax = daily.plot(figsize=(11, 4))
basin_ref[cols].resample("1D").mean().plot(ax=ax, lw=4, alpha=0.25, legend=False,
                                            color=[l.get_color() for l in ax.lines])
ax.set_ylabel("daily mean [W/m2]")
ax.set_title("thin: GEOtopPY, thick: C++", fontsize=9);

## ARF_1D: an arctic snowpack

In [ ]:
py, ref = pair("ARF_1D")
cols = ["Psnow_over_canopy[mm]", "snow_depth[mm]", "snow_water_equivalent[mm]",
        "snow_density[kg/m3]", "snow_temperature[C]", "Tsurface[C]"]
fig, axes = plt.subplots(len(cols), 1, figsize=(11, 12), sharex=True)
for ax, col in zip(axes, cols):
    ax.plot(ref.index, ref[col], lw=4, alpha=0.3, label="C++")
    ax.plot(py.index, py[col], "o-", ms=3, lw=1, label="GEOtopPY")
    ax.set_ylabel(col, fontsize=8)
axes[0].legend()
fig.tight_layout()

In [ ]:
arf = out["ARF_1D"]
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
depth_heatmap(axes[0], profile_df(arf / "soiltemp0001.txt"), "soil temperature [C]",
              "coolwarm", max_depth=2000)
depth_heatmap(axes[1], profile_df(arf / "thetaice0001.txt"), "ice content [-]",
              "Purples", max_depth=2000)
fig.tight_layout()